<a href="https://www.kaggle.com/code/izzarsulynashrudin/pcqm4mv2?scriptVersionId=309025267" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# PCQM4Mv2 

Fokus eksperimen:
- target: **HOMO-LUMO energy gap (eV)**
- model: **SGDRegressor**
- fitur: **Sparse Morgan Fingerprint**
- split eksperimen: **train** dan **valid**

In [ ]:
!pip -q install ogb rdkit

In [ ]:
import os
import gc
import time
import math
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from scipy.sparse import csr_matrix
from scipy.stats import pearsonr, spearmanr

from sklearn.linear_model import SGDRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    explained_variance_score,
    median_absolute_error,
    max_error,
)

from joblib import dump

from rdkit import Chem, RDLogger
from rdkit.Chem import (
    Descriptors,
    Lipinski,
    rdMolDescriptors,
    Crippen,
    rdFingerprintGenerator,
)

from ogb.lsc import PCQM4Mv2Dataset, PCQM4Mv2Evaluator

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")

## 1. Konfigurasi

In [ ]:
ROOT = "/kaggle/working/ogb_data"
OUTPUT_DIR = "/kaggle/working/output"
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
MODEL_DIR = "/kaggle/working/model_artifacts"

SEED = 42
DEBUG = False

EDA_SAMPLE_SIZE = 100_000 if not DEBUG else 5_000
TRAIN_BATCH_SIZE = 50_000 if not DEBUG else 10_000
PRED_BATCH_SIZE = 50_000 if not DEBUG else 10_000

FINGERPRINT_RADIUS = 2
FINGERPRINT_BITS = 2048

N_EPOCHS = 2 if not DEBUG else 1
MAX_TRAIN_ROWS = None if not DEBUG else 200_000
MAX_VALID_ROWS = None if not DEBUG else 50_000

SAVE_MODEL = True
PLOT_DPI = 160

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(PLOT_DIR).mkdir(parents=True, exist_ok=True)
Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)

## 2. Helper utilitas JSON dan helper simpan plot

In [ ]:
PLOT_REGISTRY = []

def sanitize_for_json(obj):
    if isinstance(obj, dict):
        return {str(k): sanitize_for_json(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [sanitize_for_json(v) for v in obj]
    if isinstance(obj, tuple):
        return [sanitize_for_json(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj

def save_json(filename, payload):
    path = Path(OUTPUT_DIR) / filename
    with open(path, "w", encoding="utf-8") as f:
        json.dump(sanitize_for_json(payload), f, ensure_ascii=False, indent=2)
    print(f"Saved JSON: {path}")
    return path

def register_plot(filename, title, category, description):
    entry = {
        "file_name": filename,
        "relative_path": f"plots/{filename}",
        "absolute_path": str(Path(PLOT_DIR) / filename),
        "title": title,
        "category": category,
        "description": description,
    }
    PLOT_REGISTRY.append(entry)
    return entry

def save_current_plot(filename, title, category, description, dpi=PLOT_DPI):
    full_path = Path(PLOT_DIR) / filename
    plt.tight_layout()
    plt.savefig(full_path, dpi=dpi, bbox_inches="tight")
    register_plot(filename, title, category, description)
    print(f"Saved plot: {full_path}")
    return full_path

def save_plot_registry():
    return save_json("plot_manifest.json", PLOT_REGISTRY)

def to_numpy_index(x):
    if hasattr(x, "cpu") and hasattr(x, "numpy"):
        return x.cpu().numpy()
    return np.asarray(x)

def batch_index_iterator(index_array, batch_size):
    for start in range(0, len(index_array), batch_size):
        yield index_array[start:start + batch_size]

def subset_index(idx, max_rows=None, shuffle=False, seed=SEED):
    idx = np.asarray(idx)
    if shuffle:
        rng_local = np.random.default_rng(seed)
        idx = idx.copy()
        rng_local.shuffle(idx)
    if max_rows is not None:
        idx = idx[:max_rows]
    return idx

## 3. Load dataset dan split

In [ ]:
print("Loading PCQM4Mv2 dataset in SMILES mode...")
dataset = PCQM4Mv2Dataset(root=ROOT, only_smiles=True)
evaluator = PCQM4Mv2Evaluator()

split_dict = dataset.get_idx_split()

train_idx = to_numpy_index(split_dict["train"])
valid_idx = to_numpy_index(split_dict["valid"])

other_split_counts = {}
for k, v in split_dict.items():
    if k not in ["train", "valid"]:
        other_split_counts[k] = int(len(to_numpy_index(v)))

smiles_all = np.asarray(dataset.graphs, dtype=object)
y_all = np.asarray(dataset.labels, dtype=np.float32)

dataset_overview = {
    "dataset_name": "PCQM4Mv2",
    "task_type": "graph_regression",
    "target_name": "HOMO-LUMO energy gap",
    "target_unit": "eV",
    "input_representation": "SMILES",
    "feature_representation": "sparse Morgan fingerprint",
    "total_rows": int(len(dataset)),
    "official_split_counts": {
        "train": int(len(train_idx)),
        "valid": int(len(valid_idx)),
        **other_split_counts
    },
    "experiment_splits_used": ["train", "valid"],
    "notes": "Eksperimen notebook ini hanya melatih dan mengevaluasi pada split train dan valid."
}

display(pd.DataFrame({
    "split": list(dataset_overview["official_split_counts"].keys()) + ["total"],
    "n_rows": list(dataset_overview["official_split_counts"].values()) + [len(dataset)]
}))

save_json("dataset_overview.json", dataset_overview)

## 4. Plot jumlah data per split

In [ ]:
split_plot_df = pd.DataFrame({
    "split": list(dataset_overview["official_split_counts"].keys()),
    "n_rows": list(dataset_overview["official_split_counts"].values())
})

color_map = {
    "train": "#1f77b4",
    "valid": "#ff7f0e",
    "test-dev": "#2ca02c",
    "test-challenge": "#d62728",
}

plt.figure(figsize=(9, 5))
bar_colors = [color_map.get(s, "#7f7f7f") for s in split_plot_df["split"]]
bars = plt.bar(split_plot_df["split"], split_plot_df["n_rows"], color=bar_colors)
plt.title("Jumlah Data per Split")
plt.xlabel("Split")
plt.ylabel("Jumlah Data")
for bar, val in zip(bars, split_plot_df["n_rows"]):
    plt.text(bar.get_x() + bar.get_width() / 2, val, f"{int(val):,}", ha="center", va="bottom", fontsize=9)
save_current_plot(
    "01_split_counts.png",
    "Jumlah Data per Split",
    "dataset",
    "Bar chart jumlah data untuk setiap split resmi dataset."
)
plt.show()

## 5. EDA - ekstraksi descriptor dasar

In [ ]:
def smiles_basic_features(smiles: str):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return {
            "smiles_len": len(str(smiles)),
            "num_atoms": np.nan,
            "num_bonds": np.nan,
            "heavy_atoms": np.nan,
            "hetero_atoms": np.nan,
            "mol_wt": np.nan,
            "logp": np.nan,
            "tpsa": np.nan,
            "h_donors": np.nan,
            "h_acceptors": np.nan,
            "rotatable_bonds": np.nan,
            "ring_count": np.nan,
            "aromatic_rings": np.nan,
            "valid_mol": 0,
        }

    return {
        "smiles_len": len(str(smiles)),
        "num_atoms": mol.GetNumAtoms(),
        "num_bonds": mol.GetNumBonds(),
        "heavy_atoms": Lipinski.HeavyAtomCount(mol),
        "hetero_atoms": Lipinski.NumHeteroatoms(mol),
        "mol_wt": Descriptors.MolWt(mol),
        "logp": Crippen.MolLogP(mol),
        "tpsa": rdMolDescriptors.CalcTPSA(mol),
        "h_donors": Lipinski.NumHDonors(mol),
        "h_acceptors": Lipinski.NumHAcceptors(mol),
        "rotatable_bonds": Lipinski.NumRotatableBonds(mol),
        "ring_count": rdMolDescriptors.CalcNumRings(mol),
        "aromatic_rings": rdMolDescriptors.CalcNumAromaticRings(mol),
        "valid_mol": 1,
    }

## 6. EDA - sampling dan simpan JSON

In [ ]:
rng = np.random.default_rng(SEED)
eda_sample_idx = rng.choice(
    train_idx,
    size=min(EDA_SAMPLE_SIZE, len(train_idx)),
    replace=False
)

eda_rows = []
for idx in tqdm(eda_sample_idx, desc="EDA descriptors"):
    feats = smiles_basic_features(smiles_all[idx])
    feats["target"] = float(y_all[idx])
    eda_rows.append(feats)

eda_df = pd.DataFrame(eda_rows)

display(eda_df.head())
display(eda_df.describe(include="all").T)

eda_summary = {
    "sample_size": int(len(eda_df)),
    "valid_molecule_ratio": float(eda_df["valid_mol"].mean()),
    "target_summary": {
        "mean": float(eda_df["target"].mean()),
        "std": float(eda_df["target"].std()),
        "min": float(eda_df["target"].min()),
        "median": float(eda_df["target"].median()),
        "max": float(eda_df["target"].max()),
    },
    "numeric_summary_table": eda_df.describe().reset_index().to_dict(orient="records"),
}

save_json("eda_summary.json", eda_summary)
save_json("eda_head.json", eda_df.head(20).to_dict(orient="records"))

## 7. Plot EDA

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(eda_df["target"].dropna(), bins=60)
plt.title("Distribusi Target (HOMO-LUMO gap)")
plt.xlabel("Target")
plt.ylabel("Count")
save_current_plot(
    "02_target_distribution.png",
    "Distribusi Target",
    "eda",
    "Histogram distribusi target HOMO-LUMO energy gap."
)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()
plot_cols = ["smiles_len", "num_atoms", "num_bonds", "mol_wt"]

for ax, col in zip(axes, plot_cols):
    ax.hist(eda_df[col].dropna(), bins=50)
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel("Count")

save_current_plot(
    "03_basic_descriptor_distributions.png",
    "Distribusi Descriptor Dasar",
    "eda",
    "Histogram beberapa descriptor dasar: smiles_len, num_atoms, num_bonds, dan mol_wt."
)
plt.show()

In [ ]:
num_cols = [
    "target",
    "smiles_len",
    "num_atoms",
    "num_bonds",
    "heavy_atoms",
    "hetero_atoms",
    "mol_wt",
    "logp",
    "tpsa",
    "h_donors",
    "h_acceptors",
    "rotatable_bonds",
    "ring_count",
    "aromatic_rings",
]

corr = eda_df[num_cols].corr(numeric_only=True)

plt.figure(figsize=(12, 10))
plt.imshow(corr, aspect="auto")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.colorbar()
plt.title("Correlation Heatmap (EDA Sample)")
save_current_plot(
    "04_correlation_heatmap.png",
    "Correlation Heatmap",
    "eda",
    "Heatmap korelasi antar target dan descriptor numerik."
)
plt.show()

In [ ]:
plot_df = eda_df[["mol_wt", "target"]].dropna().sample(
    min(15000, len(eda_df[["mol_wt", "target"]].dropna())),
    random_state=SEED
)

plt.figure(figsize=(7, 5))
plt.scatter(plot_df["mol_wt"], plot_df["target"], s=5, alpha=0.3)
plt.title("Target vs Molecular Weight")
plt.xlabel("Molecular Weight")
plt.ylabel("Target")
save_current_plot(
    "05_target_vs_molwt.png",
    "Target vs Molecular Weight",
    "eda",
    "Scatter plot hubungan target dan molecular weight pada sampel EDA."
)
plt.show()

In [ ]:
eda_box_cols = ["target", "num_atoms", "num_bonds", "mol_wt"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for ax, col in zip(axes, eda_box_cols):
    ax.boxplot(eda_df[col].dropna().values, vert=True)
    ax.set_title(f"Boxplot {col}")
    ax.set_ylabel(col)

save_current_plot(
    "06_boxplots_eda.png",
    "Boxplot EDA",
    "eda",
    "Boxplot beberapa variabel penting untuk melihat sebaran dan outlier."
)
plt.show()

## 8. Preprocessing - sparse Morgan fingerprint

In [ ]:
fp_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=FINGERPRINT_RADIUS,
    fpSize=FINGERPRINT_BITS,
)

def smiles_batch_to_csr(smiles_batch, fp_gen, n_bits):
    data = []
    indices = []
    indptr = [0]
    invalid = 0

    for smi in smiles_batch:
        mol = Chem.MolFromSmiles(str(smi))
        if mol is None:
            invalid += 1
            indptr.append(len(indices))
            continue

        fp = fp_gen.GetFingerprint(mol)
        onbits = list(fp.GetOnBits())
        indices.extend(onbits)
        data.extend([1.0] * len(onbits))
        indptr.append(len(indices))

    X = csr_matrix(
        (
            np.asarray(data, dtype=np.float32),
            np.asarray(indices, dtype=np.int32),
            np.asarray(indptr, dtype=np.int64),
        ),
        shape=(len(smiles_batch), n_bits),
        dtype=np.float32,
    )
    return X, invalid

preprocessing_config = {
    "input_mode": "SMILES",
    "fingerprint_type": "Morgan fingerprint",
    "fingerprint_radius": int(FINGERPRINT_RADIUS),
    "fingerprint_bits": int(FINGERPRINT_BITS),
    "matrix_type": "CSR sparse matrix",
    "feature_count_before_training": int(FINGERPRINT_BITS),
}

save_json("preprocessing_config.json", preprocessing_config)
preprocessing_config

## 9. Siapkan split eksperimen

In [ ]:
train_idx_used = subset_index(train_idx, max_rows=MAX_TRAIN_ROWS, shuffle=True, seed=SEED)
valid_idx_used = subset_index(valid_idx, max_rows=MAX_VALID_ROWS, shuffle=False)

split_used = {
    "train_original_rows": int(len(train_idx)),
    "valid_original_rows": int(len(valid_idx)),
    "train_used_rows": int(len(train_idx_used)),
    "valid_used_rows": int(len(valid_idx_used)),
    "train_rows_dropped_due_to_limit": int(len(train_idx) - len(train_idx_used)),
    "valid_rows_dropped_due_to_limit": int(len(valid_idx) - len(valid_idx_used)),
    "debug_mode": bool(DEBUG),
    "max_train_rows_setting": None if MAX_TRAIN_ROWS is None else int(MAX_TRAIN_ROWS),
    "max_valid_rows_setting": None if MAX_VALID_ROWS is None else int(MAX_VALID_ROWS),
}

display(pd.DataFrame([split_used]))
save_json("split_used.json", split_used)

In [ ]:
split_used_plot_df = pd.DataFrame({
    "category": ["train_original", "train_used", "valid_original", "valid_used"],
    "n_rows": [
        split_used["train_original_rows"],
        split_used["train_used_rows"],
        split_used["valid_original_rows"],
        split_used["valid_used_rows"],
    ]
})

plt.figure(figsize=(10, 5))
bars = plt.bar(split_used_plot_df["category"], split_used_plot_df["n_rows"])
plt.title("Perbandingan Data Original vs Dipakai")
plt.xlabel("Kategori")
plt.ylabel("Jumlah Data")
for bar, val in zip(bars, split_used_plot_df["n_rows"]):
    plt.text(bar.get_x() + bar.get_width() / 2, val, f"{int(val):,}", ha="center", va="bottom", fontsize=9)
save_current_plot(
    "07_original_vs_used_split_counts.png",
    "Original vs Used Split Counts",
    "dataset",
    "Bar chart perbandingan jumlah data original dan jumlah data yang dipakai."
)
plt.show()

## 10. Definisikan model

In [ ]:
model = SGDRegressor(
    loss="huber",
    penalty="l2",
    alpha=1e-6,
    max_iter=1,
    tol=None,
    shuffle=False,
    learning_rate="optimal",
    random_state=SEED,
    average=True,
)

model_config = {
    "model_name": "SGDRegressor",
    "model_family": "linear_regression_optimized_with_stochastic_gradient_descent",
    "feature_type": "sparse Morgan fingerprint",
    "hyperparameters": model.get_params(),
    "training_scheme": {
        "fit_mode": "incremental_batch_training_with_partial_fit",
        "epochs": int(N_EPOCHS),
        "train_batch_size": int(TRAIN_BATCH_SIZE),
        "prediction_batch_size": int(PRED_BATCH_SIZE),
        "random_seed": int(SEED),
    },
}

save_json("model_config.json", model_config)
model_config

## 11. Training

In [ ]:
history_rows = []
start_train = time.time()

for epoch in range(1, N_EPOCHS + 1):
    epoch_idx = train_idx_used.copy()
    np.random.default_rng(SEED + epoch).shuffle(epoch_idx)

    pbar = tqdm(
        batch_index_iterator(epoch_idx, TRAIN_BATCH_SIZE),
        total=math.ceil(len(epoch_idx) / TRAIN_BATCH_SIZE),
        desc=f"Epoch {epoch}"
    )

    for batch_no, batch_ids in enumerate(pbar, start=1):
        batch_smiles = smiles_all[batch_ids]
        batch_y = y_all[batch_ids]

        X_batch, invalid_batch = smiles_batch_to_csr(
            batch_smiles, fp_generator, FINGERPRINT_BITS
        )

        model.partial_fit(X_batch, batch_y)
        pred_batch = model.predict(X_batch)

        batch_mae = mean_absolute_error(batch_y, pred_batch)
        batch_rmse = np.sqrt(mean_squared_error(batch_y, pred_batch))

        history_rows.append({
            "epoch": epoch,
            "batch": batch_no,
            "rows": len(batch_ids),
            "invalid_smiles": invalid_batch,
            "train_mae": batch_mae,
            "train_rmse": batch_rmse,
        })

        pbar.set_postfix({
            "train_mae": f"{batch_mae:.4f}",
            "train_rmse": f"{batch_rmse:.4f}",
        })

        del X_batch, pred_batch, batch_smiles, batch_y
        gc.collect()

history_df = pd.DataFrame(history_rows)

training_summary = {
    "train_minutes": (time.time() - start_train) / 60.0,
    "num_epochs_completed": int(N_EPOCHS),
    "num_training_steps": int(len(history_df)),
    "last_batch_train_mae": float(history_df["train_mae"].iloc[-1]),
    "last_batch_train_rmse": float(history_df["train_rmse"].iloc[-1]),
    "best_batch_train_mae": float(history_df["train_mae"].min()),
    "best_batch_train_rmse": float(history_df["train_rmse"].min()),
}

display(history_df.tail())
save_json("training_history.json", history_df.to_dict(orient="records"))
save_json("training_summary.json", training_summary)

## 12. Hitung jumlah parameter

In [ ]:
coef_array = np.asarray(model.coef_, dtype=np.float64)
intercept_array = np.atleast_1d(model.intercept_).astype(np.float64)

parameter_summary = {
    "parameter_count_definition": "total_parameters = number_of_coefficients + number_of_intercept_terms",
    "num_input_features": int(coef_array.size),
    "num_coefficients": int(coef_array.size),
    "num_intercepts": int(intercept_array.size),
    "total_parameters": int(coef_array.size + intercept_array.size),
    "nonzero_coefficients": int(np.count_nonzero(coef_array)),
    "zero_coefficients": int(coef_array.size - np.count_nonzero(coef_array)),
    "coefficient_density": float(np.count_nonzero(coef_array) / max(coef_array.size, 1)),
    "intercept_values": intercept_array.tolist(),
    "coefficient_summary": {
        "coef_min": float(coef_array.min()),
        "coef_max": float(coef_array.max()),
        "coef_mean": float(coef_array.mean()),
        "coef_std": float(coef_array.std()),
        "coef_l1_norm": float(np.abs(coef_array).sum()),
        "coef_l2_norm": float(np.sqrt(np.square(coef_array).sum())),
    },
}

save_json("parameter_summary.json", parameter_summary)
parameter_summary

## 13. Plot training

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(np.arange(1, len(history_df) + 1), history_df["train_mae"])
plt.title("Training MAE per Batch")
plt.xlabel("Batch Step")
plt.ylabel("MAE")
save_current_plot(
    "08_training_mae_curve.png",
    "Training MAE Curve",
    "training",
    "Kurva MAE training per batch."
)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(np.arange(1, len(history_df) + 1), history_df["train_rmse"])
plt.title("Training RMSE per Batch")
plt.xlabel("Batch Step")
plt.ylabel("RMSE")
save_current_plot(
    "09_training_rmse_curve.png",
    "Training RMSE Curve",
    "training",
    "Kurva RMSE training per batch."
)
plt.show()

In [ ]:
epoch_summary_df = history_df.groupby("epoch", as_index=False).agg({
    "train_mae": "mean",
    "train_rmse": "mean",
    "invalid_smiles": "sum",
    "rows": "sum"
})

plt.figure(figsize=(8, 5))
plt.plot(epoch_summary_df["epoch"], epoch_summary_df["train_mae"], marker="o", label="Epoch MAE")
plt.plot(epoch_summary_df["epoch"], epoch_summary_df["train_rmse"], marker="o", label="Epoch RMSE")
plt.title("Rata-rata Metrik per Epoch")
plt.xlabel("Epoch")
plt.ylabel("Metric Value")
plt.legend()
save_current_plot(
    "10_epoch_metric_summary.png",
    "Epoch Metric Summary",
    "training",
    "Ringkasan rata-rata MAE dan RMSE per epoch."
)
plt.show()

## 14. Inference validation

In [ ]:
def predict_in_batches(model, index_array, batch_size, desc="Predict"):
    preds = []
    invalid_total = 0

    for batch_ids in tqdm(
        batch_index_iterator(index_array, batch_size),
        total=math.ceil(len(index_array) / batch_size),
        desc=desc
    ):
        X_batch, invalid_batch = smiles_batch_to_csr(
            smiles_all[batch_ids],
            fp_generator,
            FINGERPRINT_BITS
        )
        pred_batch = model.predict(X_batch).astype(np.float32)
        preds.append(pred_batch)
        invalid_total += invalid_batch

        del X_batch, pred_batch
        gc.collect()

    return np.concatenate(preds), invalid_total

valid_y = y_all[valid_idx_used]
valid_pred, valid_invalid = predict_in_batches(
    model,
    valid_idx_used,
    PRED_BATCH_SIZE,
    desc="Validation inference"
)

validation_info = {
    "validation_rows": int(len(valid_y)),
    "invalid_smiles_during_validation": int(valid_invalid),
    "prediction_dtype": str(valid_pred.dtype),
}

save_json("validation_info.json", validation_info)
validation_info

## 15. Evaluasi regresi lengkap

In [ ]:
def regression_report(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    errors = y_true - y_pred
    abs_errors = np.abs(errors)
    sq_errors = np.square(errors)

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    evs = explained_variance_score(y_true, y_pred)
    mxerr = max_error(y_true, y_pred)

    y_mean = np.mean(y_true)
    sse = float(np.sum(sq_errors))
    sae = float(np.sum(abs_errors))
    mbe = float(np.mean(y_pred - y_true))
    me = float(np.mean(errors))
    median_error = float(np.median(errors))
    error_std = float(np.std(errors))
    var_true = float(np.var(y_true))
    std_true = float(np.std(y_true))
    mean_true = float(np.mean(y_true))
    range_true = float(np.max(y_true) - np.min(y_true))
    iqr_true = float(np.percentile(y_true, 75) - np.percentile(y_true, 25))
    rae = float(sae / (np.sum(np.abs(y_true - y_mean)) + eps))
    rrse = float(np.sqrt(sse / (np.sum(np.square(y_true - y_mean)) + eps)))
    nrmse_mean = float(rmse / (abs(mean_true) + eps))
    nrmse_range = float(rmse / (range_true + eps))
    nrmse_std = float(rmse / (std_true + eps))
    cv_rmse_percent = float((rmse / (abs(mean_true) + eps)) * 100.0)
    wmape_percent = float((np.sum(abs_errors) / (np.sum(np.abs(y_true)) + eps)) * 100.0)
    mape_percent = float(np.mean(abs_errors / np.clip(np.abs(y_true), eps, None)) * 100.0)
    smape_percent = float(np.mean(2.0 * abs_errors / (np.abs(y_true) + np.abs(y_pred) + eps)) * 100.0)

    try:
        pearson_corr = float(pearsonr(y_true, y_pred)[0])
    except Exception:
        pearson_corr = np.nan

    try:
        spearman_corr = float(spearmanr(y_true, y_pred)[0])
    except Exception:
        spearman_corr = np.nan

    ogb_mae = float(evaluator.eval({
        "y_true": y_true.astype(np.float32),
        "y_pred": y_pred.astype(np.float32)
    })["mae"])

    metrics = [
        {"metric": "OGB_MAE", "value": ogb_mae, "category": "official"},
        {"metric": "MAE", "value": float(mae), "category": "absolute_error"},
        {"metric": "MedianAE", "value": float(medae), "category": "absolute_error"},
        {"metric": "RAE", "value": rae, "category": "relative_error"},
        {"metric": "WMAPE_percent", "value": wmape_percent, "category": "percentage_error"},
        {"metric": "MAPE_percent", "value": mape_percent, "category": "percentage_error"},
        {"metric": "sMAPE_percent", "value": smape_percent, "category": "percentage_error"},
        {"metric": "MSE", "value": float(mse), "category": "squared_error"},
        {"metric": "RMSE", "value": float(rmse), "category": "squared_error"},
        {"metric": "NRMSE_mean", "value": nrmse_mean, "category": "normalized_error"},
        {"metric": "NRMSE_range", "value": nrmse_range, "category": "normalized_error"},
        {"metric": "NRMSE_std", "value": nrmse_std, "category": "normalized_error"},
        {"metric": "CV_RMSE_percent", "value": cv_rmse_percent, "category": "normalized_error"},
        {"metric": "RRSE", "value": rrse, "category": "relative_error"},
        {"metric": "MaxError", "value": float(mxerr), "category": "extreme_error"},
        {"metric": "MeanError", "value": me, "category": "bias"},
        {"metric": "MeanBiasError_pred_minus_true", "value": mbe, "category": "bias"},
        {"metric": "MedianError", "value": median_error, "category": "bias"},
        {"metric": "ErrorStd", "value": error_std, "category": "distribution"},
        {"metric": "R2", "value": float(r2), "category": "goodness_of_fit"},
        {"metric": "ExplainedVariance", "value": float(evs), "category": "goodness_of_fit"},
        {"metric": "Pearson_r", "value": pearson_corr, "category": "correlation"},
        {"metric": "Spearman_rho", "value": spearman_corr, "category": "correlation"},
        {"metric": "SSE", "value": sse, "category": "aggregate_error"},
        {"metric": "SAE", "value": sae, "category": "aggregate_error"},
        {"metric": "Variance_y_true", "value": var_true, "category": "target_distribution"},
        {"metric": "Std_y_true", "value": std_true, "category": "target_distribution"},
        {"metric": "Mean_y_true", "value": mean_true, "category": "target_distribution"},
        {"metric": "Range_y_true", "value": range_true, "category": "target_distribution"},
        {"metric": "IQR_y_true", "value": iqr_true, "category": "target_distribution"},
    ]
    return pd.DataFrame(metrics)

metrics_df = regression_report(valid_y, valid_pred)
display(metrics_df)

prediction_preview_df = pd.DataFrame({
    "row_id": np.arange(min(200, len(valid_y))),
    "y_true": valid_y[:200].astype(float),
    "y_pred": valid_pred[:200].astype(float),
})
prediction_preview_df["error"] = prediction_preview_df["y_true"] - prediction_preview_df["y_pred"]
prediction_preview_df["abs_error"] = np.abs(prediction_preview_df["error"])

save_json("evaluation_metrics.json", metrics_df.to_dict(orient="records"))
save_json("validation_preview.json", prediction_preview_df.to_dict(orient="records"))

## 16. Plot hasil validasi

In [ ]:
plot_n = min(50000, len(valid_y))
plot_idx = np.random.default_rng(SEED).choice(
    np.arange(len(valid_y)),
    size=plot_n,
    replace=False
)

plot_true = valid_y[plot_idx]
plot_pred = valid_pred[plot_idx]
residuals = plot_true - plot_pred
abs_residuals = np.abs(residuals)

plt.figure(figsize=(7, 7))
plt.scatter(plot_true, plot_pred, s=5, alpha=0.25)
low = min(plot_true.min(), plot_pred.min())
high = max(plot_true.max(), plot_pred.max())
plt.plot([low, high], [low, high], linestyle="--")
plt.title("Validation: Predicted vs True")
plt.xlabel("True")
plt.ylabel("Predicted")
save_current_plot(
    "11_predicted_vs_true.png",
    "Predicted vs True",
    "evaluation",
    "Scatter plot nilai prediksi melawan nilai aktual pada validation set."
)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=60)
plt.title("Residual Distribution")
plt.xlabel("Residual (True - Predicted)")
plt.ylabel("Count")
save_current_plot(
    "12_residual_distribution.png",
    "Residual Distribution",
    "evaluation",
    "Histogram distribusi residual."
)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(plot_true, residuals, s=5, alpha=0.25)
plt.axhline(0.0, linestyle="--")
plt.title("Residual vs True")
plt.xlabel("True")
plt.ylabel("Residual")
save_current_plot(
    "13_residual_vs_true.png",
    "Residual vs True",
    "evaluation",
    "Scatter plot residual terhadap nilai aktual."
)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(plot_true, bins=60, alpha=0.6, label="True")
plt.hist(plot_pred, bins=60, alpha=0.6, label="Pred")
plt.title("Validation Distribution: True vs Pred")
plt.xlabel("Target")
plt.ylabel("Count")
plt.legend()
save_current_plot(
    "14_true_vs_pred_distribution.png",
    "True vs Pred Distribution",
    "evaluation",
    "Perbandingan distribusi nilai aktual dan prediksi."
)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(plot_pred, residuals, s=5, alpha=0.25)
plt.axhline(0.0, linestyle="--")
plt.title("Residual vs Predicted")
plt.xlabel("Predicted")
plt.ylabel("Residual")
save_current_plot(
    "15_residual_vs_predicted.png",
    "Residual vs Predicted",
    "evaluation",
    "Scatter plot residual terhadap nilai prediksi."
)
plt.show()

In [ ]:
sorted_abs_residuals = np.sort(abs_residuals)
cdf = np.arange(1, len(sorted_abs_residuals) + 1) / len(sorted_abs_residuals)

plt.figure(figsize=(8, 5))
plt.plot(sorted_abs_residuals, cdf)
plt.title("Cumulative Distribution of Absolute Error")
plt.xlabel("Absolute Error")
plt.ylabel("Cumulative Probability")
save_current_plot(
    "16_absolute_error_cdf.png",
    "Absolute Error CDF",
    "evaluation",
    "Kurva distribusi kumulatif absolute error."
)
plt.show()

In [ ]:
quantiles = [0.5, 0.75, 0.9, 0.95, 0.99]
quantile_values = [float(np.quantile(abs_residuals, q)) for q in quantiles]

plt.figure(figsize=(8, 5))
bars = plt.bar([str(q) for q in quantiles], quantile_values)
plt.title("Absolute Error Quantiles")
plt.xlabel("Quantile")
plt.ylabel("Absolute Error")
for bar, val in zip(bars, quantile_values):
    plt.text(bar.get_x() + bar.get_width() / 2, val, f"{val:.4f}", ha="center", va="bottom", fontsize=9)
save_current_plot(
    "17_absolute_error_quantiles.png",
    "Absolute Error Quantiles",
    "evaluation",
    "Bar chart quantile absolute error."
)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.boxplot(abs_residuals, vert=False)
plt.title("Boxplot of Absolute Error")
plt.xlabel("Absolute Error")
save_current_plot(
    "18_absolute_error_boxplot.png",
    "Absolute Error Boxplot",
    "evaluation",
    "Boxplot absolute error untuk melihat pusat dan outlier error."
)
plt.show()

In [ ]:
metrics_plot_df = metrics_df.copy()
metrics_plot_df = metrics_plot_df[
    metrics_plot_df["metric"].isin(["OGB_MAE", "MAE", "RMSE", "R2", "ExplainedVariance", "Pearson_r", "Spearman_rho"])
].copy()

plt.figure(figsize=(10, 5))
bars = plt.bar(metrics_plot_df["metric"], metrics_plot_df["value"])
plt.title("Selected Evaluation Metrics")
plt.xlabel("Metric")
plt.ylabel("Value")
plt.xticks(rotation=30)
for bar, val in zip(bars, metrics_plot_df["value"]):
    plt.text(bar.get_x() + bar.get_width() / 2, val, f"{val:.4f}", ha="center", va="bottom", fontsize=8)
save_current_plot(
    "19_selected_metrics_bar.png",
    "Selected Metrics",
    "evaluation",
    "Bar chart metrik evaluasi utama untuk presentasi."
)
plt.show()

In [ ]:
residual_sample = residuals[:min(5000, len(residuals))]
plt.figure(figsize=(10, 4))
plt.plot(np.arange(len(residual_sample)), residual_sample)
plt.title("Residual Sequence Sample")
plt.xlabel("Sample Index")
plt.ylabel("Residual")
save_current_plot(
    "20_residual_sequence_sample.png",
    "Residual Sequence Sample",
    "evaluation",
    "Plot urutan residual pada sebagian sampel validation."
)
plt.show()

## 17. Simpan model, manifest plot, dan laporan lengkap

In [ ]:
if SAVE_MODEL:
    model_path = Path(MODEL_DIR) / "sgd_morgan_pcqm4mv2.joblib"
    dump(model, model_path)
    print(f"Saved model to: {model_path}")
else:
    model_path = None

save_plot_registry()

run_summary = {
    "dataset_name": "PCQM4Mv2",
    "model_name": "SGDRegressor",
    "feature_type": "sparse Morgan fingerprint",
    "target_name": "HOMO-LUMO energy gap",
    "target_unit": "eV",
    "rows_train_used": int(len(train_idx_used)),
    "rows_valid_used": int(len(valid_idx_used)),
    "fingerprint_radius": int(FINGERPRINT_RADIUS),
    "fingerprint_bits": int(FINGERPRINT_BITS),
    "epochs": int(N_EPOCHS),
    "train_batch_size": int(TRAIN_BATCH_SIZE),
    "pred_batch_size": int(PRED_BATCH_SIZE),
    "total_parameters": int(parameter_summary["total_parameters"]),
    "nonzero_coefficients": int(parameter_summary["nonzero_coefficients"]),
    "saved_model_path": None if model_path is None else str(model_path),
    "plot_count": int(len(PLOT_REGISTRY)),
}

full_report = {
    "dataset_overview": dataset_overview,
    "split_used": split_used,
    "preprocessing_config": preprocessing_config,
    "model_config": model_config,
    "parameter_summary": parameter_summary,
    "training_summary": training_summary,
    "training_history_last_rows": history_df.tail(20).to_dict(orient="records"),
    "validation_info": validation_info,
    "evaluation_metrics": metrics_df.to_dict(orient="records"),
    "validation_preview": prediction_preview_df.to_dict(orient="records"),
    "plot_manifest": PLOT_REGISTRY,
    "run_summary": run_summary,
}

save_json("run_summary.json", run_summary)
save_json("full_report.json", full_report)

print("All key JSON artifacts and plots have been saved to OUTPUT_DIR.")
run_summary